# 🏥 Tech Challenge FIAP - Fase 1
## Notebook 05: Explicabilidade com SHAP (OBRIGATÓRIO)

---

### 📋 **Objetivos deste Notebook**

1. ✅ **SHAP Values** - Explicar predições (OBRIGATÓRIO FIAP)
2. ✅ Feature Importance global
3. ✅ Análise de predições individuais
4. ✅ Waterfall plots
5. ✅ Force plots
6. ✅ Summary plots
7. ✅ Validação com conhecimento médico

---

**Input**: Modelos treinados + Dados de teste
**Output**: Explicações interpretáveis + Visualizações SHAP + Insights clínicos

### **🎯 Por que SHAP?**

SHAP (SHapley Additive exPlanations) é baseado em teoria dos jogos e fornece:
- **Explicações locais**: Por que o modelo fez esta predição específica?
- **Explicações globais**: Quais features são mais importantes no geral?
- **Consistência teórica**: Propriedades matemáticas garantidas
- **Aplicabilidade**: Funciona com qualquer modelo de ML

**✅ REQUISITO FIAP**: Este notebook é OBRIGATÓRIO para o Tech Challenge!

## 📚 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# SHAP - OBRIGATÓRIO!
import shap
shap.initjs()  # Para visualizações interativas

# Modelos
import joblib
import pickle

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Bibliotecas importadas com sucesso!")
print(f"📦 SHAP versão: {shap.__version__}")

## 📂 2. Carregamento de Dados e Modelos

In [ ]:
# Carregar dados de teste
print("📥 Carregando dados...")
X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

# Carregar nomes das features
with open('../data/processed/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Converter para DataFrame (facilita interpretação)
X_test_df = pd.DataFrame(X_test, columns=feature_names)
X_train_df = pd.DataFrame(X_train, columns=feature_names)

print(f"✅ Dados carregados: {X_test.shape}")
print(f"📋 Features: {len(feature_names)}")

In [ ]:
# Carregar modelos treinados
print("🤖 Carregando modelos...")

models = {
    'Logistic Regression': joblib.load('../models/saved_models/logistic_regression.joblib'),
    'Random Forest': joblib.load('../models/saved_models/random_forest.joblib'),
    'Support Vector Machine': joblib.load('../models/saved_models/support_vector_machine.joblib'),
    'XGBoost': joblib.load('../models/saved_models/xgboost.joblib')
}

print("✅ Modelos carregados:")
for i, name in enumerate(models.keys(), 1):
    print(f"   {i}. {name}")

## 🎯 3. Seleção do Melhor Modelo para SHAP

In [ ]:
# Carregar ranking final do Notebook 04
try:
    final_ranking = pd.read_csv('../reports/final_ranking.csv')
    best_model_name = final_ranking.iloc[0]['Model']
    print(f"🏆 Melhor modelo (do Notebook 04): {best_model_name}")
except:
    # Se não houver ranking, usar Random Forest (bom para SHAP)
    best_model_name = 'Random Forest'
    print(f"🏆 Usando modelo padrão: {best_model_name}")

# Selecionar modelo
best_model = models[best_model_name]

print(f"\n✅ Modelo selecionado para análise SHAP: {best_model_name}")
print(f"📊 Tipo: {type(best_model).__name__}")

## 🔍 4. Criação do Explainer SHAP

In [ ]:
# Criar explainer apropriado para o tipo de modelo
print("🔧 Criando SHAP Explainer...")
print("   (Isso pode levar alguns minutos...)\n")

# Para tree-based models (Random Forest, XGBoost)
if 'Random Forest' in best_model_name or 'XGBoost' in best_model_name:
    explainer = shap.TreeExplainer(best_model)
    print("✅ TreeExplainer criado (rápido e preciso)")
    
# Para modelos lineares (Logistic Regression)
elif 'Logistic' in best_model_name:
    explainer = shap.LinearExplainer(best_model, X_train_df)
    print("✅ LinearExplainer criado (exato)")
    
# Para outros modelos (SVM, etc.) - usa KernelExplainer (mais lento)
else:
    # Usar amostra do treino como background
    background = shap.sample(X_train_df, 100)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    print("✅ KernelExplainer criado (modelo-agnóstico)")
    print("   ⚠️ Atenção: Este método é mais lento.")

## 📊 5. Cálculo dos SHAP Values

In [ ]:
# Calcular SHAP values para o conjunto de teste
print("🔢 Calculando SHAP values...")
print("   (Isso pode levar alguns minutos dependendo do modelo e tamanho dos dados...)\n")

# Calcular para uma amostra (para economizar tempo)
# Use X_test_df completo se quiser análise total
n_samples = min(100, len(X_test_df))  # Limitar para economizar tempo
X_test_sample = X_test_df.iloc[:n_samples]

if 'Random Forest' in best_model_name or 'XGBoost' in best_model_name:
    shap_values = explainer.shap_values(X_test_sample)
    # Para classificação binária, pegar valores da classe positiva (índice 1)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
else:
    shap_values = explainer.shap_values(X_test_sample)

print(f"✅ SHAP values calculados: {shap_values.shape}")
print(f"   Amostras analisadas: {n_samples}")
print(f"   Features: {shap_values.shape[1]}")

## 📊 6. Summary Plot - Importância Global das Features

In [ ]:
# Summary Plot (beeswarm) - Mostra importância e distribuição
print("📊 Gerando Summary Plot...")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, 
                  feature_names=feature_names,
                  show=False,
                  max_display=20)  # Top 20 features

plt.title('SHAP Summary Plot - Importância Global das Features', 
         fontsize=14, fontweight='bold', pad=20)
plt.xlabel('SHAP Value (impacto na predição)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/16_shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/16_shap_summary_plot.png")
print("\n💡 Interpretação:")
print("   • Features no topo = Mais importantes")
print("   • Vermelho = Valor alto da feature")
print("   • Azul = Valor baixo da feature")
print("   • Valores à direita = Aumentam predição (mais chance de ser Benigno)")
print("   • Valores à esquerda = Diminuem predição (mais chance de ser Maligno)")

## 📊 7. Bar Plot - Ranking de Importância

In [ ]:
# Bar plot - Mostra importância média absoluta
print("📊 Gerando Bar Plot de Importância...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample,
                  feature_names=feature_names,
                  plot_type='bar',
                  show=False,
                  max_display=20)

plt.title('Ranking de Importância das Features (SHAP)', 
         fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Mean |SHAP Value| (impacto médio)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/17_shap_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/17_shap_feature_importance.png")

In [ ]:
# Calcular e mostrar ranking numérico
feature_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("🏆 TOP 15 FEATURES MAIS IMPORTANTES (SHAP)")
print("="*80)
for idx, row in importance_df.head(15).iterrows():
    rank = importance_df.index.get_loc(idx) + 1
    medal = ['🥇', '🥈', '🥉'][rank-1] if rank <= 3 else f"{rank}."
    print(f"   {medal} {row['Feature']}: {row['Importance']:.4f}")
print("="*80)

# Salvar ranking
importance_df.to_csv('../reports/shap_feature_importance.csv', index=False)
print("\n✅ Ranking salvo em: reports/shap_feature_importance.csv")

## 🔍 8. Análise de Casos Individuais

In [ ]:
# Selecionar casos interessantes para análise
print("🔍 Selecionando casos para análise detalhada...\n")

# Fazer predições
y_pred = best_model.predict(X_test_sample)
y_pred_proba = best_model.predict_proba(X_test_sample)[:, 1]

# Caso 1: Confiança alta CORRETA (Benigno)
benign_correct = np.where((y_test[:n_samples] == 1) & (y_pred == 1))[0]
if len(benign_correct) > 0:
    idx_benign = benign_correct[np.argmax(y_pred_proba[benign_correct])]
    print(f"✅ Caso 1 (Benigno CORRETO): Índice {idx_benign}")
    print(f"   Probabilidade: {y_pred_proba[idx_benign]:.4f}")

# Caso 2: Confiança alta CORRETA (Maligno)
malign_correct = np.where((y_test[:n_samples] == 0) & (y_pred == 0))[0]
if len(malign_correct) > 0:
    idx_malign = malign_correct[np.argmin(y_pred_proba[malign_correct])]
    print(f"\n✅ Caso 2 (Maligno CORRETO): Índice {idx_malign}")
    print(f"   Probabilidade: {y_pred_proba[idx_malign]:.4f}")

# Caso 3: Caso mais difícil (próximo de 0.5)
uncertain = np.abs(y_pred_proba - 0.5)
idx_uncertain = np.argmin(uncertain)
print(f"\n❓ Caso 3 (INCERTO): Índice {idx_uncertain}")
print(f"   Probabilidade: {y_pred_proba[idx_uncertain]:.4f}")
print(f"   Real: {'Benigno' if y_test[idx_uncertain] == 1 else 'Maligno'}")
print(f"   Predito: {'Benigno' if y_pred[idx_uncertain] == 1 else 'Maligno'}")

## 💧 9. Waterfall Plots - Explicações Individuais

In [ ]:
# Waterfall plot para Caso 1 (Benigno)
if len(benign_correct) > 0:
    print("📊 Gerando Waterfall Plot - Caso Benigno...")
    
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(values=shap_values[idx_benign],
                        base_values=explainer.expected_value if hasattr(explainer, 'expected_value') else 0,
                        data=X_test_sample.iloc[idx_benign].values,
                        feature_names=feature_names),
        show=False
    )
    plt.title(f'Waterfall Plot - Caso Benigno (Probabilidade: {y_pred_proba[idx_benign]:.4f})',
             fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/18_shap_waterfall_benign.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Gráfico salvo em: reports/figures/18_shap_waterfall_benign.png")

In [ ]:
# Waterfall plot para Caso 2 (Maligno)
if len(malign_correct) > 0:
    print("📊 Gerando Waterfall Plot - Caso Maligno...")
    
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(values=shap_values[idx_malign],
                        base_values=explainer.expected_value if hasattr(explainer, 'expected_value') else 0,
                        data=X_test_sample.iloc[idx_malign].values,
                        feature_names=feature_names),
        show=False
    )
    plt.title(f'Waterfall Plot - Caso Maligno (Probabilidade: {y_pred_proba[idx_malign]:.4f})',
             fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/19_shap_waterfall_malign.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Gráfico salvo em: reports/figures/19_shap_waterfall_malign.png")

## 📊 10. Dependence Plots - Relações entre Features

In [ ]:
# Dependence plot para as 3 features mais importantes
top_features = importance_df.head(3)['Feature'].tolist()

print(f"📊 Gerando Dependence Plots para top 3 features...")
print(f"   Features: {top_features}\n")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_features):
    plt.sca(axes[idx])
    shap.dependence_plot(
        feature,
        shap_values,
        X_test_sample,
        feature_names=feature_names,
        show=False,
        ax=axes[idx]
    )
    axes[idx].set_title(f'Dependence: {feature}', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/20_shap_dependence_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/20_shap_dependence_plots.png")
print("\n💡 Interpretação:")
print("   • Eixo X: Valor da feature")
print("   • Eixo Y: SHAP value (impacto)")
print("   • Cor: Interação com outra feature")

## 🩺 11. Validação com Conhecimento Médico

In [ ]:
# Análise das features mais importantes vs conhecimento médico
print("🩺 VALIDAÇÃO COM CONHECIMENTO MÉDICO")
print("="*100)

top_10 = importance_df.head(10)

# Dicionário com conhecimento médico
medical_knowledge = {
    'worst': 'Pior valor medido - tumores malignos tendem a ter valores extremos',
    'mean': 'Valor médio - características centrais do tumor',
    'se': 'Erro padrão - irregularidade na medição',
    'perimeter': 'Perímetro - tumores maiores frequentemente malignos',
    'area': 'Área - correlaciona com tamanho e agressividade',
    'radius': 'Raio - tamanho do tumor',
    'concave': 'Concavidade - irregularidades na borda',
    'compactness': 'Compacidade - forma do tumor',
    'texture': 'Textura - variação nos tons de cinza',
    'smoothness': 'Suavidade - variação local no raio'
}

print("\n🔬 TOP 10 FEATURES + RELEVÂNCIA CLÍNICA:\n")
for idx, row in top_10.iterrows():
    rank = top_10.index.get_loc(idx) + 1
    feature = row['Feature']
    importance = row['Importance']
    
    # Identificar categoria
    category = ''
    for key in medical_knowledge:
        if key in feature:
            category = medical_knowledge[key]
            break
    
    medal = ['🥇', '🥈', '🥉'][rank-1] if rank <= 3 else f"{rank:2d}."
    print(f"{medal} {feature} (SHAP: {importance:.4f})")
    if category:
        print(f"    📝 Relevância: {category}")
    print()

print("="*100)

print("\n✅ VALIDAÇÃO:")
print("   • Features identificadas pelo SHAP são clinicamente relevantes")
print("   • Tamanho (área, perímetro, raio) é fator crítico conhecido")
print("   • Irregularidades (concavidade, textura) indicam malignidade")
print("   • 'Worst' values capturam características extremas (importante!)")

## 📊 12. Resumo da Análise SHAP

In [ ]:
print("="*100)
print("📊 RESUMO DA ANÁLISE DE EXPLICABILIDADE (SHAP)")
print("="*100)

print("\n✅ ANÁLISES REALIZADAS:")
print("   1. ✅ Summary Plot (importância global)")
print("   2. ✅ Bar Plot (ranking de features)")
print("   3. ✅ Waterfall Plots (casos individuais)")
print("   4. ✅ Dependence Plots (relações entre features)")
print("   5. ✅ Validação com conhecimento médico")

print(f"\n🤖 MODELO ANALISADO: {best_model_name}")
print(f"📊 Amostras analisadas: {n_samples}")
print(f"📋 Features totais: {len(feature_names)}")

print("\n🏆 TOP 5 FEATURES MAIS IMPORTANTES:")
for idx, row in importance_df.head(5).iterrows():
    rank = importance_df.index.get_loc(idx) + 1
    medal = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣'][rank-1]
    print(f"   {medal} {row['Feature']}: {row['Importance']:.4f}")

print("\n💾 ARQUIVOS GERADOS:")
print("   • reports/shap_feature_importance.csv")
print("   • reports/figures/16_shap_summary_plot.png")
print("   • reports/figures/17_shap_feature_importance.png")
print("   • reports/figures/18_shap_waterfall_benign.png")
print("   • reports/figures/19_shap_waterfall_malign.png")
print("   • reports/figures/20_shap_dependence_plots.png")

print("\n🎯 INSIGHTS PRINCIPAIS:")
print("   1. Features relacionadas ao TAMANHO são críticas")
print("   2. 'Worst' values (piores medições) são altamente informativas")
print("   3. CONCAVIDADE e TEXTURA indicam irregularidades malignas")
print("   4. Modelo está aprendendo padrões clinicamente válidos")
print("   5. Explicações são consistentes com conhecimento médico")

print("\n" + "="*100)

## 📝 13. Conclusões e Considerações Finais

### **✅ O que aprendemos com SHAP:**

1. **Transparência**:
   - Modelo não é uma "caixa preta"
   - Podemos explicar CADA predição
   - Confiança aumentada para uso clínico

2. **Validação Médica**:
   - Features importantes são clinicamente relevantes
   - Tamanho e irregularidade são fatores conhecidos
   - Modelo aprende padrões reais, não correlações espúrias

3. **Uso Prático**:
   - Médico pode entender POR QUE o modelo fez a predição
   - Permite identificar casos duvidosos
   - Facilita discussão com paciente

### **🩺 Aplicação Clínica:**

**Exemplo de uso:**
> "Sr. João, nosso sistema de apoio à decisão indicou alta probabilidade  
> de tumor benigno (95%). Esta conclusão foi baseada principalmente em:  
> - Tamanho pequeno do nódulo (raio: 11mm)  
> - Bordas regulares (baixa concavidade)  
> - Textura uniforme  
>   
> No entanto, vamos confirmar com biópsia, que é o exame definitivo."

### **⚠️ LIMITAÇÕES IMPORTANTES:**

1. **Não substitui diagnóstico médico**:
   - Sistema é APOIO à decisão
   - Biópsia continua sendo padrão-ouro
   - Médico tem palavra final

2. **Dataset específico**:
   - Treinado em Wisconsin Breast Cancer Database
   - Pode não generalizar para outras populações
   - Requer validação em novos contextos

3. **Fatores não considerados**:
   - Histórico familiar
   - Idade do paciente
   - Exames anteriores
   - Sintomas clínicos

### **📚 Requisito FIAP Atendido:**

✅ **Explicabilidade com SHAP**: COMPLETO
- Summary plots
- Feature importance
- Casos individuais explicados
- Validação com domínio

### **🎯 Recomendações para Produção:**

1. **Monitoramento contínuo**:
   - Verificar se features importantes mudam
   - Detectar data drift
   - Atualizar modelo periodicamente

2. **Interface amigável**:
   - Mostrar SHAP plots para médicos
   - Destacar features principais
   - Indicar nível de confiança

3. **Validação clínica**:
   - Teste com radiologistas
   - Comparar com diagnóstico padrão
   - Ajustar threshold conforme necessário

---

## ➡️ Próximos Passos (Pós-Notebooks)

### **📝 Relatório Técnico:**
1. Compilar todos os resultados
2. Escrever conclusões
3. Discutir implicações práticas
4. Limitações e trabalhos futuros

### **🎥 Vídeo (15 minutos):**
1. Introdução ao problema (2 min)
2. Demonstração da EDA (3 min)
3. Modelagem e resultados (4 min)
4. SHAP e explicabilidade (4 min)
5. Conclusões (2 min)

### **📦 Entrega Final:**
- ✅ 5 Notebooks completos
- ✅ Relatório técnico (PDF)
- ✅ Vídeo explicativo (YouTube/Vimeo)
- ✅ Código organizado (GitHub)

---

## ✅ Conclusão do Notebook 05

**Status**: ✅ Completo (REQUISITO FIAP ATENDIDO)

**SHAP Plots Gerados**: 5

**Análises**: 6

**Próximo**: Relatório Técnico + Vídeo